# Reduction and Parallel Streams

CSC-239 · Module 10 · Lesson 4 of 4

You can transform and filter collection entries, complete a stream once, and create fresh streams for additional computations. This lesson combines processed values into one result and explains the conditions for safe parallel processing.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Choose a valid identity and associative operation for reduction.
- Build and test equivalent sequential and parallel pipelines without shared mutable accumulation.


## Why This Matters

A report may need one total rather than a list of every entry. The combining rule must still work when a system divides the input into parts and combines the partial results.


## Check Your Starting Point

Recall how an accumulator loop starts a total and updates it for each selected value. Explain why a second stream computation needs a fresh stream. Identify the side effect when two references update one shared mutable object.

**My explanation:**


## Concept

### Reduce elements to one result

A **reduction** combines processed elements into one result. The reduce operation used here takes a starting value and a combining operation.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(3);
counts.add(5);
int total = counts.stream().reduce(0, (left, right) -> left + right);
System.out.println("Total: " + total);
```

This prints Total: 8. The combining lambda receives two values and returns their sum. Parentheses enclose the two parameter names. The expression uses the same addition you used in accumulator loops.

This form of reduce expects a BinaryOperator: a functional interface whose two inputs and result have the same type. That is the functional-interface pattern from Lesson 1 applied to two inputs. The Integer elements and the lambda's addition use the boxing and unboxing rules already practiced in Module 4.

### Choose a neutral starting value

An **identity value** is a neutral starting value that does not change the combined result. For addition, zero is the identity because 0 + value equals value and value + 0 equals value.

```java
import java.util.ArrayList;
ArrayList<Integer> empty = new ArrayList<Integer>();
int total = empty.stream().reduce(0, (left, right) -> left + right);
System.out.println("Empty total: " + total);
```

This prints Empty total: 0. With no elements to combine, the identity is the result. Testing empty input checks a real part of the reduction's contract.

Starting addition with one would add an extra value in a sequential computation. It would also violate the identity requirement used when partial results are combined. Do not select a starting value only because it produces a desired answer for one input.

### Check the grouping rule

An **associative combination** gives the same result when its inputs are regrouped. Addition of the small integers in this lesson is associative:

```text
(2 + 4) + 1 = 7
2 + (4 + 1) = 7
```

Changing grouping is different from choosing a starting value. A correct reduction needs both the identity and the required combining behavior.

Subtraction fails the grouping check:

```text
(8 - 3) - 1 = 4
8 - (3 - 1) = 6
```

Because regrouping changes the result, subtraction is not a valid associative combining operation for this reduction contract. One observed parallel result cannot repair that mismatch. Do not memorize one output from an invalid parallel reduction as if it were guaranteed.

Use small integers whose totals fit in int for these exercises. Other number types need their own grouping checks.

### Combine independent parts in parallel

**Parallel stream processing** allows different parts of the pipeline work to run at the same time, then combines their partial results. parallelStream requests that mode for a collection source. Java manages the work.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(3);
counts.add(5);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Both totals are 8. A neutral identity and associative addition allow partial totals to be combined correctly. The two terminal operations use separate fresh streams.

The two println calls occur after each terminal operation completes, in the ordinary order written by the caller. They do not claim that individual elements were processed in a fixed order. Source encounter order and the schedule of parallel execution are different ideas.

Parallel processing is not a promise of a speed improvement. Dividing work and combining results requires extra work. Small tasks may take longer, and available computing resources affect the outcome. Correctness is the goal of these exercises; elapsed time is not a grading condition.

### Keep each rule independent

**Stateless behavior** means a processing rule does not depend on changing shared state. A lambda such as score -> score * 2 derives its result from its input. It does not need a shared counter or a result from the preceding element.

**Noninterference** means leaving the stream source unchanged while its pipeline executes. Do not add to or remove from the source inside its map, filter, or reduction operation.

A shared mutable total would introduce a different problem: several pieces of work could try to read and change the same value. The reduce operation already supplies a way to combine values under its contract. Keep the arithmetic in the combining lambda and let the operation combine partial results.

Do not hide a shared mutable total inside an array or object to bypass the local-variable capture restriction. Capturing an unchanged reference does not make updates to its object safe for parallel use.

### Combine the stages into a complete task

A pipeline can select values, transform them, and then reduce them. Explain the meaning of each stage and its input before choosing parallel execution.

For the independent task, negative and zero scores fail the positive-score filter. Each retained positive score is doubled. Addition combines the doubled values, with zero as the identity. Repeated positive scores remain separate contributions; the pipeline keeps every matching entry.

Check at least these cases: empty input, input with no retained values, one retained value, and repeated retained values. Predict the sequential and parallel totals before running either version. If they disagree, investigate the operation contract and shared state instead of accepting whichever result appeared first.


## Video Demonstration

Watch the same input receive separate sequential and parallel reductions. Predict the totals and identify why zero and addition satisfy the combining contract.

<video controls preload="metadata" width="960">
  <source src="media/04_reduction_and_parallel_streams/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/04_reduction_and_parallel_streams/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the reduction and parallel streams demonstration transcript](media/04_reduction_and_parallel_streams/transcript.md).


## Worked Example

**Subgoal 1: choose the reduction rule.** Use zero as the identity and addition to combine counts.

**Subgoal 2: build separate computations.** Create one fresh sequential stream and one fresh parallel stream.

**Subgoal 3: compare complete results.** Print only after the terminal operations have returned.


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(2);
counts.add(4);
counts.add(1);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


Expected output:

```text
Sequential: 7
Parallel: 7
```

Both computations combine two, four, and one to produce seven. Zero does not change a partial sum, and regrouping addition preserves the total. No lambda changes shared state or modifies the source.


## Predict, Run, Trace, and Explain

### Predict two reductions

Predict both printed lines before running. Explain how the zero entry and the zero identity each affect the result. Identify the two inputs to the combining lambda and justify why the sequential and parallel results should agree for this operation.

My predicted lines:

Effect of the identity and the zero source entry:

The combining lambda’s two inputs and result:

Why the two totals should agree:


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


Run the complete prediction program in the Workspace. Keep your original prediction and explain every confirmation or correction. Identify where each fresh stream is created and where reduction returns its result. Explain why the printed order is known even though it does not establish an order for parallel element processing.

My original prediction:

Actual output:

My post-run explanation:

Fresh stream and terminal operation in each computation:

What the two print statements do and do not show:

### Trace the combination contract

Trace a sequential sum starting with the identity and visiting the three entries in list order. Then write two different groupings of five, zero and six that keep their order, and compare their totals. Explain why checking the identity differs from checking grouping. Identify whether the source or a shared total is changed by either combining rule. State what you can conclude about correctness and what you cannot conclude about speed from the two matching results.

| Step | Incoming value | Total after combination |
| --- | --- | --- |
| Start with identity | | |
| First entry | | |
| Second entry | | |
| Third entry | | |
Two valid groupings:

Identity requirement versus grouping requirement:

State and source changes, if any:

Correctness evidence and limits on speed claims:

<details>
<summary>Show answer</summary>

Both reductions combine five, zero and six to produce eleven. The identity zero contributes no extra amount, including when it is combined with a partial result. Addition of these small integers gives the same total under regrouping. Each reduction receives a fresh stream: stream requests sequential processing, while parallelStream requests parallel processing. The two-parameter lambda returns the sum of its arguments without updating a shared total or changing counts. The caller prints Sequential: 11 and then Parallel: 11 after the corresponding results have been computed. These print statements do not reveal the processing order of individual elements. A sequential trace can start at zero and combine to obtain five, then five, then eleven. For example, grouping five with zero first or zero with six first still produces eleven. This is a grouping check, not a claim about how the parallel implementation divided this run. The BinaryOperator pattern matches two Integer inputs and an Integer result; the familiar boxing and unboxing rules support the arithmetic. Matching totals confirm these results, but do not prove a speed improvement.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 11
Parallel: 11
```

Common error: Adding the identity as an extra nonzero contribution. Assuming parallel processing means a different mathematical operation. Treating equal totals as proof of a particular element-processing order.

</details>


### Check neutrality and regrouping directly

Before running, predict all six lines. The plus value is a `BinaryOperator<Integer>`: apply invokes its combining rule with two Integer inputs and returns an Integer. Explain which calls check the identity and which check grouping. Compare the two subtraction expressions. After running, keep your predictions and explain what the subtraction counterexample says about using subtraction as this reduction’s combining rule.

My six predicted lines:

Actual output and my corrections:

Identity checks and grouping checks:

Why the subtraction results matter:


In [ ]:
import java.util.function.BinaryOperator;
BinaryOperator<Integer> plus = (left, right) -> left + right;
System.out.println("Left identity: " + plus.apply(0, 6));
System.out.println("Right identity: " + plus.apply(6, 0));
System.out.println("Left grouping: " + plus.apply(plus.apply(6, 2), 3));
System.out.println("Right grouping: " + plus.apply(6, plus.apply(2, 3)));
System.out.println("Left subtraction: " + ((9 - 4) - 2));
System.out.println("Right subtraction: " + (9 - (4 - 2)));


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The first two calls return six because zero is neutral on either side of addition. The next calls combine six, two and three with different groupings and both return eleven. The plus value has type `BinaryOperator<Integer>`: its apply call accepts two Integer inputs and returns an Integer result. The subtraction expressions return three and seven because regrouping subtraction changes its result. That counterexample shows why subtraction cannot satisfy the associative rule required here. These are direct arithmetic checks; the program does not run an invalid parallel reduction or claim a fixed result for one.

```java
import java.util.function.BinaryOperator;
BinaryOperator<Integer> plus = (left, right) -> left + right;
System.out.println("Left identity: " + plus.apply(0, 6));
System.out.println("Right identity: " + plus.apply(6, 0));
System.out.println("Left grouping: " + plus.apply(plus.apply(6, 2), 3));
System.out.println("Right grouping: " + plus.apply(6, plus.apply(2, 3)));
System.out.println("Left subtraction: " + ((9 - 4) - 2));
System.out.println("Right subtraction: " + (9 - (4 - 2)));
```

Expected output:

```text
Left identity: 6
Right identity: 6
Left grouping: 11
Right grouping: 11
Left subtraction: 3
Right subtraction: 7
```

Common error: Treating a neutral identity as sufficient without checking grouping. Changing the order of values when asked only to regroup them. Memorizing one result of an invalid parallel operation instead of checking its contract.

</details>


### Check independent rules and the unchanged source

Predict the complete output before running. Trace the value returned by map for each source entry and the result of reduction. After running, explain why the source values printed afterward still match the original entries. Assess two proposed edits in words: making the lambda use and update one shared changing total, and making it add an entry to counts while the pipeline runs. Identify the statelessness or noninterference requirement each would violate. Explain why the output does not reveal the parallel processing order or prove a speed improvement.

My predicted totals and source lines:

Actual output and post-run explanation:

Mapped value for each original entry:

Why each proposed edit violates a requirement:

What the printed lines cannot establish:


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(4);
counts.add(1);
counts.add(2);
int sequential = counts.stream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
for (int value : counts) {
    System.out.println("Source: " + value);
}


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

Mapping produces five, two and three, which sum to ten in either computation. The lambda derives each mapped value from its own input, and the combining lambda derives its result from its two arguments. Neither updates a shared total. The source remains four, one and two, as its later printed entries show. A proposed rule that depends on a changing shared total violates stateless behavior. A proposed addition to counts from a running pipeline operation violates noninterference. The caller prints only after computing the totals and then reads the source list; those lines do not show which parallel element was processed first. Matching totals also provide no evidence that parallel processing was faster.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(4);
counts.add(1);
counts.add(2);
int sequential = counts.stream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(value -> value + 1)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
for (int value : counts) {
    System.out.println("Source: " + value);
}
```

Expected output:

```text
Sequential: 10
Parallel: 10
Source: 4
Source: 1
Source: 2
```

Common error: Treating a copied reference to a shared mutable object as independent state. Adding to the source from a running processing rule. Using the order of later caller prints as evidence of element scheduling.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the paired reductions

Replace SEQUENTIAL_SOURCE, PARALLEL_SOURCE, IDENTITY and COMBINE. Use stream and parallelStream for the two fresh sources, and choose the neutral value and arithmetic operation for addition. Replace each repeated placeholder consistently. Copy the whole completed program into the work cell and run it. Explain why both computations need the same valid identity and combining operation.

This sample is for repair:

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.SEQUENTIAL_SOURCE().reduce(IDENTITY, (left, right) -> left COMBINE right);
int parallel = counts.PARALLEL_SOURCE().reduce(IDENTITY, (left, right) -> left COMBINE right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```


My four replacements:

Actual output:

Why the operation and identity agree across both computations:

<details>
<summary>Show answer</summary>

Use stream, parallelStream, zero and the plus operator for the four placeholders. Each source method creates a fresh computation, and each reduce uses the addition contract. Repeating the same valid identity and operation lets independently combined parts produce the same result. Both reductions combine five, zero and six to produce eleven. The identity zero contributes no extra amount, including when it is combined with a partial result. Addition of these small integers gives the same total under regrouping. Each reduction receives a fresh stream: stream requests sequential processing, while parallelStream requests parallel processing. The two-parameter lambda returns the sum of its arguments without updating a shared total or changing counts. The caller prints Sequential: 11 and then Parallel: 11 after the corresponding results have been computed. These print statements do not reveal the processing order of individual elements.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 11
Parallel: 11
```

Common error: Using a nonneutral starting value to force one desired result. Changing the arithmetic in only one pipeline. Leaving a placeholder in the executable program.

</details>


### Transform each count before reducing

Add a map operation before reduce in both pipelines so each count is multiplied by three. Predict the transformed values and both totals, then run your complete program. Next replace the zero source entry with negative two, keeping both pipelines unchanged, and test again. Explain why the negative value contributes in this program and why the mapping needs no shared changing total.


In [ ]:
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);


My transformed values and predicted totals:

Actual first output:

My prediction and output after the negative input change:

Why the negative contribution remains:

<details>
<summary>Show answer</summary>

The map operation changes five, zero and six into fifteen, zero and eighteen. Addition with identity zero combines them to thirty-three in both pipelines. After replacing the source zero with negative two, mapping produces fifteen, negative six and eighteen, totaling twenty-seven. This program has no filtering stage, so the negative contribution remains. Each mapping result depends only on its input; reduce combines returned values instead of requiring a shared variable.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 33
Parallel: 33
```

Common error: Adding the mapping stage to only one of the two computations. Multiplying the identity instead of each processed input. Assuming a negative input is automatically filtered out.

**Additional test: `Replace the zero source entry with negative two`.** The mapped negative input contributes negative six because there is no filter. Both complete pipelines sum fifteen, negative six and eighteen to twenty-seven with zero as the identity.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(-2);
counts.add(6);
int sequential = counts.stream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream()
    .map(count -> count * 3)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 27
Parallel: 27
```

</details>


### Repair a nonneutral starting value

The displayed sequential program runs, but it claims to add only the supplied counts. Predict its printed value and locate the extra contribution. Explain why its starting value is not an identity for addition. Repair the starting value and include a separate fresh parallel reduction using the same correct rule, with Sequential and Parallel output labels. Put your whole repaired program in the work cell and run it. Then remove all source additions and test the complete repaired pair again. Explain what the empty result establishes about the identity.

This sample is for repair:

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(1, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
```


My predicted faulty output and extra contribution:

Why the starting value is invalid:

Actual output after the paired repair:

Prediction and output for empty repaired input:

What the empty case demonstrates:

<details>
<summary>Show answer</summary>

The faulty sequential calculation starts at one and then adds five, zero and six, so it prints Sequential: 12. The extra one is not a source contribution and is not neutral: one plus five is six, not five. A starting value must satisfy the identity rule, not merely fit one example. Restoring zero gives eleven for both complete repaired reductions. With all source additions removed, both repaired reductions return zero. That empty-input result is the identity itself. The faulty draft is sequential only; no fixed result for an invalid parallel reduction is asserted.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
counts.add(5);
counts.add(0);
counts.add(6);
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 11
Parallel: 11
```

Common error: Changing the expected total to accept the extra starting contribution. Repairing one mode while leaving an invalid identity in the other. Claiming one observed invalid reduction result establishes a valid contract.

**Additional test: `Empty source with the repaired identity`.** No input values are available to combine. Both complete valid reductions return their neutral zero identity, using separate fresh streams.

```java
import java.util.ArrayList;
ArrayList<Integer> counts = new ArrayList<Integer>();
int sequential = counts.stream().reduce(0, (left, right) -> left + right);
int parallel = counts.parallelStream().reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 0
Parallel: 0
```

</details>


## Independent Practice

### Build the positive-score total

Create scores with the four Integer values 4, -2, 0 and 7, in order. Keep only positive scores, double each retained score, and reduce with identity zero and addition. Build separate fresh sequential and parallel pipelines with the same stages. Print `Sequential: 22` and `Parallel: 22` on separate lines. Include the import, complete source setup and both computations. Explain the input and output of each stage, why the identity and grouping rule are valid, and why a shared mutable total is unnecessary.

My complete program:

Actual baseline output:

Retained values, mapped values and final total:

Identity and grouping explanation:

How the rules avoid shared mutable accumulation and source changes:


### Check empty, rejected, single and repeated values

Test four separate source lists: empty input; -3, 0 and -1; 3, 3, 0 and -2; and the single value 5. Keep the filtering, mapping, identity, combining rule and two output statements unchanged. Predict retained values, mapped values and both totals before each run. Compare the actual results and explain why empty input differs from a nonempty source with no retained values, why both repeated positives contribute, and what the single retained value checks. Restore and rerun the original four-score case. Judge correctness from the stage rules and results; elapsed time is not a grading condition.

For each case, my retained values, mapped values and predicted totals:

Actual empty and no-retained-value results:

Actual repeated and single-value results:

My explanations and any corrections:

Restored baseline output:


<details>
<summary>Show answer</summary>

The positive-score filter retains four and seven and rejects negative two and zero. Mapping doubles the retained values to eight and fourteen. Reduction combines them to twenty-two, starting with the neutral identity zero. Both the sequential and parallel computations use fresh streams and the same stages. The addition rule uses its two supplied values instead of updating a shared mutable total. No stage edits scores. Addition of these small values preserves the total under regrouping, so both complete results print twenty-two. Empty input and input with only nonpositive scores both leave no values to reduce, so each returns zero. Their sources differ, but the reduction receives no contributions in either case. The repeated-positive input retains both threes, maps each to six and totals twelve. A single five maps to ten and reduces to ten because zero does not alter it. The same unchanged rules explain both execution modes for all four cases.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(4);
scores.add(-2);
scores.add(0);
scores.add(7);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 22
Parallel: 22
```

Common error: Doubling values without applying the required positive filter. Removing repeated matching entries even though each is a contribution. Updating a separate shared total instead of returning values for reduce.

**Additional test: Empty score list.** No source entries are present, so both fresh pipelines return the zero identity.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 0
Parallel: 0
```

**Additional test: Only nonpositive scores: -3, 0, -1.** The source has three entries, but the positive filter rejects each one. Both reductions receive no retained values and return zero.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(-3);
scores.add(0);
scores.add(-1);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 0
Parallel: 0
```

**Additional test: Repeated positives: 3, 3, 0, -2.** Both threes pass and each maps to six. Zero and negative two are rejected. The two separate contributions produce twelve.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(3);
scores.add(3);
scores.add(0);
scores.add(-2);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 12
Parallel: 12
```

**Additional test: One positive score: 5.** The single positive score maps to ten. Adding the neutral identity leaves ten unchanged in each complete pipeline.

```java
import java.util.ArrayList;
ArrayList<Integer> scores = new ArrayList<Integer>();
scores.add(5);
int sequential = scores.stream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
int parallel = scores.parallelStream()
    .filter(score -> score > 0)
    .map(score -> score * 2)
    .reduce(0, (left, right) -> left + right);
System.out.println("Sequential: " + sequential);
System.out.println("Parallel: " + parallel);
```

Expected output:

```text
Sequential: 10
Parallel: 10
```

</details>


## Summary

A reduction combines elements into one result. Its identity is neutral, and its combining operation must satisfy the required grouping rule. Parallel streams can combine independently processed parts when those rules hold. Stateless behavior avoids dependence on changing shared values, and noninterference leaves the source unchanged during execution. Correct parallel results do not imply a guaranteed speed improvement.

Close the answers and justify the identity, combination, source behavior, and empty-input result of your final pipeline.


## Reflection

Choose a numerical report from your field. State which values to retain, how to transform each retained value, and how to combine the results. Explain whether the combining rule is associative and what should happen when no values remain.

**My design and explanation:**

The next module writes binary data and object state to files. Those I/O streams serve a different purpose from the collection-processing streams used here.


## Supplemental Reading

- [Stream reduction contract](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/Stream.html#reduce(T,java.util.function.BinaryOperator)) defines identity and associative combination requirements.
- [Parallel pipelines and side effects](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/package-summary.html) explains stateless behavior, noninterference, and execution limits.
